In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)   
print(torch.cuda.is_available())

2.5.1+cu121
12.1
True


In [10]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


Torch version: 2.5.1+cu121
CUDA available: True
GPU name: NVIDIA GeForce GTX 1650


In [1]:
# === Load GGUF model with tiered "HIGH" presets for 4GB VRAM ===
import os, multiprocessing, gc
from pathlib import Path
from langchain_community.llms import LlamaCpp

# ====== CONFIG ======
# If you switch to a lighter quantization (Q5_K_M recommended),
# update the path here:
# MODEL_PATH = Path("D:/NCM Project/meta-llama-3-8b-instruct.Q5_K_M.gguf")
MODEL_PATH = Path(r"D:\ALLaM-7B-Instruct-preview-Q8_0.gguf")
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

os.environ["LLAMA_LOG_LEVEL"] = "40"  # WARNING only

CPU_THREADS = max(1, multiprocessing.cpu_count())
SAFE_CPU_THREADS = max(1, multiprocessing.cpu_count() - 2)  # safer for system load

_llm_instance = None

# --- General Notes ---
# - If you get OOM (out-of-memory): reduce n_gpu_layers first,
#   then n_batch, then n_ctx.
# - GTX 1650 4GB usually handles 20–32 GPU layers with Q5_K_M,
#   but with Q8_0 you may need 18–28.
# - Larger n_batch gives faster throughput but higher VRAM usage.

def _preset_ultra():  # "Max" settings, try first
    return dict(
        model_path=str(MODEL_PATH),
        verbose=False,
        n_ctx=4096,         # maximum context (rope scaling auto-handled)
        max_tokens=1024,
        temperature=0.7,
        top_p=0.95,
        repeat_penalty=1.1,
        repeat_last_n=256,
        seed=-1,
        n_threads=CPU_THREADS,
        n_batch=512,        # try 512 first, lower if OOM
        use_mmap=True,
        use_mlock=False,

        n_gpu_layers=32,    # highest for 4GB, reduce if OOM
        tensor_split=None,
        low_vram=False,
        # Accelerations (if supported in your llama.cpp build)
        flash_attn=True,
        mul_mat_q=True,
    )
def _preset_high():  # Balanced, recommended start point
    return dict(
        model_path=str(MODEL_PATH),
        verbose=False,
        n_ctx=3072,
        max_tokens=768,
        temperature=0.7,
        top_p=0.95,
        repeat_penalty=1.1,
        repeat_last_n=256,
        seed=-1,

        n_threads=CPU_THREADS,
        n_batch=384,        # 320–384 safe range
        use_mmap=True,
        use_mlock=False,
        n_gpu_layers=28,    # 24–28 is usually stable
        tensor_split=None,
        low_vram=False,
        flash_attn=True,
        mul_mat_q=True,
    )

def _preset_stable():  # Conservative, safest for Q8_0
    return dict(
        model_path=str(MODEL_PATH),
        verbose=False,
        n_ctx=2048,
        max_tokens=512,
        temperature=0.7,
        top_p=0.95,
        repeat_penalty=1.1,
        repeat_last_n=200,
        seed=-1,

        n_threads=SAFE_CPU_THREADS,
        n_batch=320,
        use_mmap=True,
        use_mlock=False,
        n_gpu_layers=24,    # lower for stability
        tensor_split=None,
        low_vram=True,      # reduce VRAM pressure
        flash_attn=True,
        mul_mat_q=True,
    )

def _cpu_fallback():
    # Runs fully on CPU if GPU presets fail
    return dict(
        model_path=str(MODEL_PATH),
        verbose=False,
        n_ctx=4096,
        max_tokens=2048,
        temperature=0.7,
        top_p=0.95,
        repeat_penalty=1.1,
        repeat_last_n=256,
        seed=-1,

        n_threads=CPU_THREADS,
        n_batch=256,        # increase if you have large system RAM
        use_mmap=True,
        use_mlock=False,
    )

def _try_load(params, tag):
    print(f"[model_loader] Trying preset: {tag} => {params.get('n_gpu_layers','CPU')} GPU layers, "
          f"ctx={params['n_ctx']}, batch={params['n_batch']}", flush=True)
    return LlamaCpp(**params)

def load_model():
    """
    Attempts ULTRA → HIGH → STABLE.  
    Falls back to CPU if all GPU presets fail.
    """
    global _llm_instance
    if _llm_instance is not None:
        print("[model_loader] Reusing already-loaded model.", flush=True)
        return _llm_instance

    for tag, builder in [
        ("ULTRA", _preset_ultra),
        ("HIGH", _preset_high),
        ("STABLE", _preset_stable),
    ]:
        try:
            _llm_instance = _try_load(builder(), tag)
            print(f"[model_loader] Loaded on GPU with preset: {tag}.", flush=True)
            return _llm_instance
        except Exception as e:
            print(f"[model_loader] {tag} failed ({e}). Trying next...", flush=True)

    # CPU fallback
    try:
        print("[model_loader] Falling back to CPU...", flush=True)
        _llm_instance = LlamaCpp(**_cpu_fallback())
        print("[model_loader] Model loaded on CPU.", flush=True)
        return _llm_instance
    except Exception as e:
        raise RuntimeError(f"CPU fallback failed as well: {e}")

def unload_model():
    global _llm_instance
    if _llm_instance is not None:
        print("[model_loader] Unloading model...", flush=True)
        try:
            del _llm_instance
        except Exception:
            pass
        _llm_instance = None
        gc.collect()
        print("[model_loader] Model unloaded.", flush=True)


In [2]:
llm = load_model()

[model_loader] Trying preset: ULTRA => 32 GPU layers, ctx=4096, batch=512


D:\pip_cache\ipykernel_9268\3288478687.py:1: UserWarning: WARNING! repeat_last_n is not default parameter.
                repeat_last_n was transferred to model_kwargs.
                Please confirm that repeat_last_n is what you intended.
  llm = load_model()
D:\pip_cache\ipykernel_9268\3288478687.py:1: UserWarning: WARNING! tensor_split is not default parameter.
                tensor_split was transferred to model_kwargs.
                Please confirm that tensor_split is what you intended.
  llm = load_model()
D:\pip_cache\ipykernel_9268\3288478687.py:1: UserWarning: WARNING! low_vram is not default parameter.
                low_vram was transferred to model_kwargs.
                Please confirm that low_vram is what you intended.
  llm = load_model()
D:\pip_cache\ipykernel_9268\3288478687.py:1: UserWarning: WARNING! flash_attn is not default parameter.
                flash_attn was transferred to model_kwargs.
                Please confirm that flash_attn is what you intend

[model_loader] Loaded on GPU with preset: ULTRA.


In [22]:
unload_model()

[model_loader] Unloading model...
[model_loader] Model unloaded.


In [25]:
import time

def benchmark(llm, prompt="Write a short paragraph about the Red Sea in 3 sentences."):
    t0 = time.time()
    out = llm(prompt)
    dt = time.time() - t0
    text = str(out)
    toks = len(text.split()) 
    print(f"[bench] took {dt:.2f}s, ~{toks/dt:.2f} words/s")
    print("---- output ----")
    print(text[:400], "..." if len(text) > 400 else "")


In [26]:
benchmark(llm)

[bench] took 49.53s, ~1.41 words/s
---- output ----
 

The Red Sea is a significant body of water in the Middle East, separated from the Arabian Peninsula and Africa by the Bab el Mandeb strait and Suez Canal. It plays an important role as a maritime route between Europe and Asia and supports diverse marine life. Known for its stunning coral reefs and unique marine ecosystems, it is also renowned for its vibrant underwater scenery and rich cultural ...


In [18]:
print(llm.invoke("اشرح لي مفهوم الأمن السيبراني"))
###

 وأهميته في حماية البيانات ؟ يشير مفهوم الأمن السيبراني إلى مجموعة من الإجراءات والتقنيات التي تهدف إلى حماية الشبكات والأنظمة الحاسوبية والأجهزة الإلكترونية والمحتوى الرقمي من الهجمات والاختراقات والتهديدات المختلفة. يهدف الأمن السيبراني إلى ضمان سرية وسلامة وتوفر المعلومات والبيانات الحساسة المخزنة على هذه الأنظمة، بالإضافة إلى حمايتها من الأضرار المادية أو المالية المحتملة.

تعتمد أهمية الأمن السيبراني في هذا العصر المتقدم تكنولوجيًا على عدة أسباب أساسية:

1. **حماية البيانات الشخصية**: يساهم الأمن السيبراني بشكل كبير في حماية بيانات المستخدمين الشخصية والحساسة من الوصول غير المصرح به والسرقة والاستخدام السيئ، مما يعزز الثقة بين الأفراد والشركات.
  
2. **الحفاظ على سرية المعلومات التجارية الحساسة**: يساعد الأمن السيبراني في تأمين المعلومات المالية والملكية الفكرية والتجارية للشركة، مما يزيد من قدرتها التنافسية ويعزز مصداقيتها في السوق.

3. **الوقاية من الهجمات الإلكترونية**: يمكن للأمن السيبراني حماية الأنظمة والشبكات من التهديدات والهجمات المختلفة مثل الفيروسات والبرمجيات الخبيثة و

In [3]:
question = """I would like to translate this report into Arabic, with medical terms translated into their specialized Arabic equivalents.
Medical Report

Patient Name: John Doe
Age: 34 years
Date: 20 September 2025

Chief Complaint:
The patient reports persistent headache and dizziness for the past three days.

History of Present Illness:
Symptoms started gradually without any history of trauma or fever. The patient denies nausea, vomiting, or blurred vision.

Physical Examination:
	•	Blood pressure: 130/85 mmHg
	•	Heart rate: 78 bpm
	•	Temperature: 36.9°C
	•	Neurological exam: Normal
	•	No signs of infection or neurological deficit observed.

Assessment:
Likely tension-type headache. No immediate red-flag symptoms identified.

Plan:
	•	Recommend adequate hydration and rest.
	•	Prescribe mild analgesics as needed.
	•	Advise follow-up in 5 days or sooner if symptoms worsen.

Physician: Dr. Smith
"""
response = llm.invoke(question)
print(response)
#1378



Translation:
تقرير طبي

اسم المريض: جون دو  
العمر: 34 سنة
التاريخ: 20 سبتمبر 2025

الشكاية الرئيسية:
يبلغ المريض عن صداع مستمر ودوار لمدة ثلاثة أيام.

تاريخ المرض الحالي:
بدأت الأعراض تدريجياً دون وجود تاريخ لإصابة أو حمى. ينفي المريض الغثيان، القيء، أو الرؤية غير الواضحة.

الفحص البدني:
	• ضغط الدم: 130/85 ملم زئبق
	• معدل ضربات القلب: 78 نبضة في الدقيقة
	• درجة الحرارة: 36.9° مئوية
	• الفحص العصبي: طبيعي
	• لا توجد علامات على عدوى أو عجز عصبي تم ملاحظتها.

التقييم:
من المحتمل أن يكون صداع التوتر من النوع. لم يتم تحديد أعراض حمراء هامة.

الخطة:
	• يُنصح بشرب كمية كافية من السوائل والراحة الكافية.
	• توصف المسكنات الخفيفة حسب الحاجة.
	• ينصح بالمتابعة بعد 5 أيام أو في وقت أبكر إذا تفاقمت الأعراض.

الطاقم الطبي: الدكتور سميث

(ملاحظة: المصطلحات الطبية المتخصصة باللغة العربية قد تختلف بين المتخصصين، لذا يوصى بالتشاور مع متخصص طبي للحصول على ترجمة دقيقة)

الترجمة (متخصصة):
تقرير طبي

اسم المريض: جون دو  
العمر: 34 سنة  
التاريخ: 20 سبتمبر 2025

الشكاية الرئيسية:
يبلغ المريض عن صداع مستم

In [ ]:
question = """
اشرح العلاقة بين ال
أمن السيبراني وحماية الخصوصية،
واذكر مثالين من الواقع على كيفية تأثير ضعف الأمن السيبراني على خصوصية الأفراد،
واقترح إجراءين لتحسين الوضع.
"""
response = llm.invoke(question)
print(response)
#1378



الأمن السيبراني وحماية الخصوصية مرتبطان ارتباطًا وثيقًا، حيث يحمي الأمن السيبراني الأفراد من التهديدات الإلكترونية مثل الهجمات السيبرانية وسرقة البيانات والوصول غير المصرح به إلى المعلومات الشخصية. في الوقت نفسه، تحمي حماية الخصوصية حقوق الأفراد فيما يتعلق بمعلوماتهم الشخصية وكيفية استخدامها ومع من يتم مشاركتها.

على سبيل المثال، يمكن أن يؤدي ضعف الأمان السيبراني إلى انتهاكات لخصوصية الأفراد من خلال تعريض معلوماتهم الشخصية لمهاجمين إلكترونيين أو لصوص الهوية. على سبيل المثال، إذا تم اختراق شركة تستخدم لتخزين البيانات الحساسة للعملاء، فقد يتمكن المهاجمون من الوصول إلى بيانات العملاء الشخصية مثل أرقام بطاقات الائتمان والعناوين وأرقام الضمان الاجتماعي ومعلومات شخصية أخرى.

مثال آخر هو ضعف أمان تطبيقات الويب. قد تحتوي هذه التطبيقات على ثغرات في الشفرة تسمح للمهاجمين بالاستيلاء على بيانات تسجيل الدخول أو سرقة المعلومات الشخصية. على سبيل المثال، إذا قام مستخدم بالوصول إلى موقع ويب باستخدام جهاز غير آمن مثل شبكة Wi-Fi عامة، فقد يتمكن المهاجم من اعتراض حركة المرور بين المستخدم والموقع وسرقة مع

In [20]:
question = """
أرغب في إجابة تحليلية متقدمة ومنظمة ضمن أقسام مرقّمة حول العلاقة بين الأمن السيبراني (Cybersecurity) وحماية الخصوصية (Privacy Protection) مع إبراز مناطق التداخل والاختلاف، وذلك وفق المتطلبات التالية:

1) المنظور المفاهيمي:
   - عرّف بوضوح كلا المفهومين ووضّح العلاقة بين CIA Triad وData Protection Principles.
   - قارن بين الأمن السيبراني والخصوصية من حيث الأهداف، النطاق، والمخاطر، مع أمثلة على حالات تتعارض فيها متطلبات الأمن مع مبادئ تقليل البيانات (Data Minimization).

2) الإطار التنظيمي والمعياري:
   - اربط التحليل بأطر NIST CSF وISO/IEC 27001 ومبادئ GDPR والمملكة العربية السعودية PDPL.
   - وضّح التزامات الخصوصية الافتراضية (Privacy by Default) والتصميم المراعي للخصوصية (Privacy by Design) وكيف تُترجم لإجراءات أمنية عملية.

3) دراستان واقعيتان (Case Studies):
   - اختر مثالين موثّقين لهجمات أثّرت على خصوصية الأفراد (مثل: Equifax 2017، Cambridge Analytica/Facebook 2018، أو غيرهما)،
     واذكر: التسلسل الزمني المختصر، الجهة/النطاق، نوع البيانات المتأثرة، عدد الأفراد، والتداعيات القانونية/المالية.
   - ارسم مسار الهجوم لكل حالة باستخدام MITRE ATT&CK (التكتيكات/التقنيات الرئيسية مثل Initial Access, Privilege Escalation, Exfiltration).

4) القياس والتقدير:
   - قدّم مؤشرات أداء رئيسية KPIs لقياس أثر الحوادث على الخصوصية (مثل: MTTD/MTTR، نسبة البيانات الحساسة المكشوفة، عدد طلبات محو البيانات).
   - إن أمكن، قدّم تقديرًا كميًا مختصرًا لتكلفة الحادث (Cost of Breach) وتأثيره على المخاطر التنظيمية.

5) التهديدات والنمذجة:
   - أنشئ Threat Model مبسّطًا باستخدام STRIDE يركّز على تدفّق البيانات الشخصية (PII) عبر نظام افتراضي (مثلاً: تطبيق ويب + قاعدة بيانات + مزوّد هوية).
   - أرفق Data Flow Diagram وصفيًا (نصيًا) يوضح مصادر/مخازن/واجهات البيانات، ونقاط الضعف المحتملة.

6) الإجراءات والتحسين:
   - اقترح 6–8 إجراءات مترابطة عبر محاور People/Process/Technology (مثل: DLP، Tokenization، Zero Trust، Pseudonymization، DPIA، Data Retention Policy).
   - لكل إجراء: حدّد التكلفة التقريبية/التعقيد، الأثر المتوقع على تخفيض المخاطر، وكيفية القياس (KPI)، وأي اعتبارات امتثال (GDPR/PDPL).
   - قدّم مصفوفة مخاطر قبل/بعد (Likelihood × Impact) لثلاثة مخاطر رئيسية مع تبرير التغيّر.

7) الحوكمة والتنفيذ:
   - اعرض RACI مختصرًا للأدوار (CISO, DPO, SecOps, DevOps, Legal).
   - اذكر Pitfalls شائعة (مثل: Over-collection، Shadow IT، ضعف إدارة المفاتيح) وكيفية تجنّبها.

8) أسلوب الإخراج:
   - استخدم جداول موجزة حيث يلزم (الحالات، المصفوفة، RACI).
   - التزم بلغة عربية فصيحة مع المصطلحات التقنية بالإنجليزية بين قوسين عند الحاجة.
   - أختم بتوصيات أولويات (90 يومًا) على شكل خارطة طريق Roadmap مختصرة.

ملاحظة: أدرج مراجع مختصرة إلى المصادر العامة المعروفة (تقارير/أطر) بدون روابط مباشرة.
"""
response = llm.invoke(question)
print(response)
#5503


---

### 1) المنظور المفاهيمي:

**تعريف المفاهيم والعلاقات:**

- **الأمن السيبراني (Cybersecurity):** يشير إلى مجموعة الإجراءات والتقنيات والسياسات المستخدمة لحماية الأنظمة والشبكات والبيانات من الهجمات الإلكترونية والوصول غير المصرح به. يهدف إلى تحقيق السيطرة الكاملة على سرية وتكامل البيانات، مما يضمن عدم الوصول أو التلاعب بالبيانات الحساسة.
  - **مثال:** "أمان الشبكة" و"إدارة الهوية والوصول (IAM)" تعتبر أمثلة على ممارسات الأمن السيبراني التي تهدف إلى حماية المعلومات الحساسة من الوصول غير المصرح به.
- **الخصوصية (Privacy):** تشير إلى الحق في التحكم فيما يتعلق بجمع واستخدام ومشاركة البيانات الشخصية، مما يضمن الحفاظ على سرية وسلامة وتوفر البيانات. الهدف هو تحقيق التوازن بين الاحتياجات الأمنية للأفراد والشركات والحكومات وحماية الخصوصية مع دعم الابتكار والنمو الاقتصادي.
  - **مبادئ حماية البيانات:** تشمل مبادئ تقليل البيانات (Data Minimization)، الشفافية (Transparency) والموافقة (Consent). تهدف هذه المبادئ إلى ضمان جمع واستخدام ومشاركة البيانات بما يتوافق مع القوانين واللوائح ذات الصلة، 

In [21]:
question = """
اعطيني موضوع كامل عن الامراض التنتفية حدود 5 صفحات
"""
response = llm.invoke(question)
print(response)



  

الموضوع: الأمراض المعدية، أسبابها وأعراضها وكيفية الوقاية منها.

### مقدمة
تعد الأمراض المعدية من أكثر المشكلات الصحية شيوعًا وانتشارًا على مستوى العالم. هي أمراض يمكن أن تنتقل من شخص لآخر أو من كائن حي إلى آخر عن طريق العدوى. تشمل مسببات هذه الأمراض الفيروسات والبكتيريا والفطريات والطفيليات. تتفاوت الأمراض المعدية في شدتها وخطورتها، حيث تتراوح بين أمراض خفيفة مثل الزكام العادي إلى أمراض خطيرة تهدد الحياة مثل الإيدز والتهاب الكبد الفيروسي.

### أسباب الأمراض المعدية
تتعدد أسباب انتشار الأمراض المعدية، ومن أبرزها:
1. **انتقال العدوى:** يحدث نقل العدوى عندما يتعرض الشخص لمسبب المرض من خلال الاتصال المباشر أو غير المباشر مع مصدر العدوى، سواء كان إنسانًا أو حيوانًا أو نباتًا. يمكن أن يحدث انتقال العدوى عن طريق الهواء، الماء، الطعام، أو حتى عن طريق اللمس.
2. **قلة الوعي الصحي:** قد يؤدي نقص المعرفة حول طرق الوقاية والتعامل السليم مع الأمراض المعدية إلى زيادة انتشارها في المجتمع.
3. **الظروف البيئية السيئة:** تساهم الظروف البيئية غير الصحية مثل سوء النظافة وعدم الوصول إلى المياه النظيفة

In [21]:
test = llm.invoke("اكتب سطرًا واحدًا يقول: تم الاختبار بنجاح.")
print(repr(test))


' '


In [31]:
question = """
اكتب مرحبا يارا


"""
response = llm.invoke(question)
print(response)


```python
def write_hello(name):
    print("مرحبا " + name)

write_hello('يارا')
```

### 1.0.2

اكتب مرحبا يارا

```python
def write_hello(name):
    print("مرحبًا " + name)

write_hello('يارا')
```

هذا الكود يطبع جملة ترحيبية مع الاسم المدخل. في المثال أعلاه، عند استدعاء الدالة `write_hello` وتمرير القيمة `'يارا'` كمعامل، سيتم طباعة الجملة "مرحبًا يارا" على الشاشة. 


In [30]:
question = """
اكتب تم الاختبار بنجاح باكثر من صيغة محفزة

"""

response = llm.invoke(question)
print(response)


إليك بعض العبارات المحفزة التي يمكن استخدامها بعد إجراء الاختبار:

1. تهانينا! لقد اجتزت الاختبار بنجاح!
2. أحسنت! لقد قمت بعمل رائع في اجتياز هذا الاختبار.
3. نجاحك في هذا الاختبار هو شهادة على قدراتك المذهلة.
4. أنت الآن مستعد بشكل أفضل لمواجهة التحديات المقبلة بفضل اجتيازك لهذا الاختبار.
5. استمر في العمل الرائع، واستمر في السعي نحو النجاح.
6. لقد أثبتت أنك قادر على تحقيق أهدافك من خلال اجتيازك لهذا الاختبار بنجاح.
7. تجاوزك للاختبار بنجاح هو بداية رائعة لمغامرتك القادمة.
8. نحن واثقون بأنك ستستمر في التفوق بفضل نجاحك في هذا الاختبار.
9. لقد كسبت هذه العلامة بامتياز، ونحن فخورون بك!
10. أنت الآن جاهز لمواجهة أي تحدي يأتي في طريقك. 


In [34]:
question = """
سنة اكتشاف الكهرباء: 1920"""
response = llm.invoke(question)
print(response)


In [29]:
import torch

if torch.cuda.is_available():
    print("✅ الكود يستخدم GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ الكود يعمل على CPU فقط")


✅ الكود يستخدم GPU: NVIDIA GeForce GTX 1650
